# DBSCAN Example (Mall Customers Dataset)

Here it is demonstrated how to use the `DBSCAN` module from the CMOR-438 library to discover density-based clusters.
In this example, the Mall Customers dataset is used and DBSCAN is compared directly to K-Means on the same 2D feature slice.

**Goal: Find clusters of shoppers by Income and Spending Score without specifying the number of clusters.**

DBSCAN advantages demonstrated here:
- Discovers the number of clusters automatically
- Labels sparse or unusual shoppers as **noise** (outliers)
- Finds non-spherical cluster shapes

## 1. Setup and Data Loading

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '../k_means_clustering')
from dbscan import DBSCAN
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv('../../../data/Mall_Customers.csv')
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_raw = mall[MALL_FEATURES].values.astype(float)
X_mall = StandardScaler().fit_transform(X_raw)

print(f"Dataset loaded: {mall.shape[0]} samples, {len(MALL_FEATURES)} features.")

## 2. Run DBSCAN

DBSCAN is applied to the 2D slice of Annual Income and Spending Score (scaled).
Parameters used:
- `eps=0.45` — neighbourhood radius (chosen via k-distance graph inspection)
- `min_samples=4` — minimum neighbours to qualify as a core point

In [ ]:
X_2feat = X_mall[:, 1:]  # Annual Income + Spending Score (scaled)

db = DBSCAN(eps=0.45, min_samples=4).fit(X_2feat)
print(f'Clusters found: {db.n_clusters_}')
print(f'Noise points:   {(db.labels_==-1).sum()}')
print(f'Core points:    {len(db.core_samples_)}')

## 3. Results and Visualisation

Two side-by-side plots compare DBSCAN and K-Means on the same Income vs Spending Score slice:
- **Left — DBSCAN:** each cluster gets its own colour; noise points (outliers) are shown as grey X markers; core points are outlined in black
- **Right — K-Means (k=5):** for comparison, showing how K-Means forces all points into clusters even when some are outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_db = plt.cm.get_cmap('tab10', db.n_clusters_)

for lbl in np.unique(db.labels_):
    mask = db.labels_ == lbl
    if lbl == -1:
        axes[0].scatter(X_raw[mask,1], X_raw[mask,2], c='lightgray', s=20, marker='x', zorder=2, label='Noise')
    else:
        axes[0].scatter(X_raw[mask,1], X_raw[mask,2], color=cmap_db(lbl), s=35, alpha=0.8, label=f'Cluster {lbl}')
axes[0].scatter(X_raw[db.core_samples_,1], X_raw[db.core_samples_,2],
                edgecolors='black', facecolors='none', s=55, linewidths=0.8, zorder=3, label='Core')
axes[0].set_xlabel('Annual Income (k$)'); axes[0].set_ylabel('Spending Score (1-100)')
axes[0].set_title(f'DBSCAN - {db.n_clusters_} clusters found', fontweight='bold')
axes[0].legend(fontsize=7)

km2 = KMeans(k=5, init='k-means++', n_init=5, random_state=42).fit(X_2feat)
for c in range(5):
    mask = km2.labels_==c
    axes[1].scatter(X_raw[mask,1], X_raw[mask,2], color=plt.cm.tab10(c), s=30, alpha=0.75, label=f'C{c}')
axes[1].set_xlabel('Annual Income (k$)'); axes[1].set_ylabel('Spending Score (1-100)')
axes[1].set_title('K-Means (k=5) - Same 2D Slice', fontweight='bold'); axes[1].legend(fontsize=7)
fig.suptitle('DBSCAN vs K-Means on Income and Spending Score', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()